In [28]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

C:\Users\color\Desktop\bank-marketing-projec


In [29]:
import pandas as pd
import numpy as np
import sys
sys.path.append("src")

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "Malgun Gothic"

from preprocessing import load_config, load_raw_data, clean_data, save_processed
from feature_engineering import run_feature_engineering


In [30]:
cfg = load_config("config/config.yaml")

In [31]:
df_raw = load_raw_data(cfg["data"]["raw_path"])
df = clean_data(df_raw)
print(df.shape)
df.head()

[INFO] 결측치 컬럼:
pdays    8324
dtype: int64
(11162, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,NaN,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,NaN,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,NaN,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,NaN,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,NaN,0,unknown,1


In [32]:
from feature_engineering import (
    add_contact_bucket, add_prev_success,
    add_pdays_contacted, add_balance_segment
)

df_fe = add_contact_bucket(df.copy())
df_fe = add_prev_success(df_fe)
df_fe = add_pdays_contacted(df_fe)
df_fe = add_balance_segment(df_fe)

new_cols = ["campaign_bucket", "prev_success", "was_contacted_before", "balance_segment"]
print(df_fe[new_cols].value_counts("campaign_bucket"))

campaign_bucket
1회      4798
2회      3028
3회      1321
4-5회    1149
6회+      866
Name: count, dtype: int64


In [35]:
df_final = run_feature_engineering(df, cfg["features"]["categorical"])
print("최종 shape:", df_final.shape)
print("컬럼 수:", len(df_final.columns))

save_processed(df_final, cfg["data"]["final_path"])

최종 shape: (11162, 52)
컬럼 수: 52
[INFO] 저장 완료: data/processed/bank_final.parquet | shape: (11162, 52)


In [34]:
num_df = df_final.select_dtypes(include=[float, int])
corr = num_df.corr()[["deposit"]].drop("deposit").sort_values("deposit", ascending=False)

print("deposit과 상관관계 상위 10개:")
print(corr.head(10))

deposit과 상관관계 상위 10개:
                       deposit
duration              0.451919
prev_success          0.286642
was_contacted_before  0.230850
pdays                 0.151593
previous              0.139867
balance               0.081129
age                   0.034901
day                  -0.056326
campaign             -0.128081
